# Semiconductor Image Restoration — NAFNet Baseline Pipeline

This notebook implements a complete, self-contained **NAFNet Baseline** pipeline for Joint $2\times$ Super-Resolution and Denoising of semiconductor inspection images.

### Dataset Properties Summary:
- **Input (NoisyLR)**: $128 \times 128$ spatial dimensions, `float32`, noisy range $\approx [-0.05, 1.41]$.
- **Target (Ground Truth GT)**: $256 \times 256$ spatial dimensions, `float32`, clean normalized range $[0.0, 1.0]$.
- **Goal**: Map noisy low-resolution inputs $(B, 1, 128, 128)$ to restored high-resolution images $(B, 1, 256, 256)$.

## 1. System Setup & Environment Check

In [ ]:
import os
import glob
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.notebook import tqdm

# Hardware Acceleration Check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Operating on device: {device}")
if device.type == 'cuda':
    print(f"[INFO] GPU Device Name: {torch.cuda.get_device_name(0)}")

# Optional Google Drive Mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[INFO] Google Drive mounted successfully.")
except Exception:
    print("[INFO] Running in local environment / non-Colab context.")

## 2. Dataset Setup / Synthetic Data Generation
If your dataset `.npy` files are located in a folder, set `LR_DIR` and `GT_DIR` below. If not present, this cell will automatically generate 50 sample paired `.npy` files for instant baseline testing.

In [ ]:
# Paths configuration
DATASET_ROOT = "dataset_sample"
LR_DIR = os.path.join(DATASET_ROOT, "NoisyLR")
GT_DIR = os.path.join(DATASET_ROOT, "GT")

def prepare_dataset_directories(lr_path, gt_path, num_samples=50):
    if not os.path.exists(lr_path) or len(glob.glob(os.path.join(lr_path, "*.npy"))) == 0:
        print(f"[INFO] Creating sample synthetic dataset at '{DATASET_ROOT}'...")
        os.makedirs(lr_path, exist_ok=True)
        os.makedirs(gt_path, exist_ok=True)
        
        for i in range(num_samples):
            # Generate synthetic 256x256 GT pattern
            x, y = np.meshgrid(np.linspace(0, 1, 256), np.linspace(0, 1, 256))
            gt_img = 0.5 + 0.4 * np.sin(10 * x + i) * np.cos(10 * y)
            gt_img = np.clip(gt_img, 0.0, 1.0).astype(np.float32)
            
            # Downsample 2x to 128x128 and add speckle + Gaussian noise
            lr_clean = gt_img[::2, ::2]
            noise = np.random.normal(0, 0.08, size=(128, 128)).astype(np.float32)
            speckle = np.random.uniform(-0.1, 0.1, size=(128, 128)).astype(np.float32)
            lr_img = lr_clean + noise + speckle
            
            np.save(os.path.join(lr_path, f"{i:06d}.npy"), lr_img)
            np.save(os.path.join(gt_path, f"{i:06d}.npy"), gt_img)
        print(f"[SUCCESS] Generated {num_samples} sample paired .npy files.")
    else:
        print(f"[INFO] Found dataset at '{lr_path}' and '{gt_path}'.")

prepare_dataset_directories(LR_DIR, GT_DIR)

## 3. PyTorch Dataset & Data Loader

In [ ]:
class PairedNpyDataset(Dataset):
    def __init__(self, lr_dir: str, gt_dir: str, augment: bool = True):
        super().__init__()
        self.lr_filenames = sorted(glob.glob(os.path.join(lr_dir, "*.npy")))
        self.gt_filenames = sorted(glob.glob(os.path.join(gt_dir, "*.npy")))
        self.augment = augment
        assert len(self.lr_filenames) == len(self.gt_filenames), "Mismatch between LR and GT files"
        self.file_pairs = list(zip(self.lr_filenames, self.gt_filenames))

    def __len__(self):
        return len(self.file_pairs)

    def _augment(self, lr, gt):
        if random.random() > 0.5:
            lr, gt = np.fliplr(lr), np.fliplr(gt)
        if random.random() > 0.5:
            lr, gt = np.flipud(lr), np.flipud(gt)
        k = random.randint(0, 3)
        if k > 0:
            lr, gt = np.rot90(lr, k=k), np.rot90(gt, k=k)
        return lr.copy(), gt.copy()

    def __getitem__(self, idx):
        lr_path, gt_path = self.file_pairs[idx]
        lr_arr = np.clip(np.load(lr_path).astype(np.float32), 0.0, 1.0)
        gt_arr = np.clip(np.load(gt_path).astype(np.float32), 0.0, 1.0)

        if self.augment:
            lr_arr, gt_arr = self._augment(lr_arr, gt_arr)

        lr_tensor = torch.from_numpy(lr_arr).unsqueeze(0)
        gt_tensor = torch.from_numpy(gt_arr).unsqueeze(0)
        return lr_tensor, gt_tensor

# Create Train & Validation Split
full_dataset = PairedNpyDataset(LR_DIR, GT_DIR, augment=True)
num_total = len(full_dataset)
num_val = max(1, int(num_total * 0.15))
num_train = num_total - num_val

train_ds, val_ds = random_split(full_dataset, [num_train, num_val], generator=torch.Generator().manual_seed(42))
val_ds.dataset.augment = False

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"[INFO] Dataset Split: {num_train} Training samples, {num_val} Validation samples.")

## 4. NAFNet Model Architecture

In [ ]:
class LayerNorm2d(nn.Module):
    def __init__(self, channels: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return self.weight.unsqueeze(-1).unsqueeze(-1) * x + self.bias.unsqueeze(-1).unsqueeze(-1)

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class NAFBlock(nn.Module):
    def __init__(self, c: int, dw_expand: int = 2, ffn_expand: int = 2):
        super().__init__()
        dw_channel = c * dw_expand
        self.conv1 = nn.Conv2d(c, dw_channel, 1)
        self.conv2 = nn.Conv2d(dw_channel, dw_channel, 3, padding=1, groups=dw_channel)
        self.sg1 = SimpleGate()
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(dw_channel // 2, dw_channel // 2, 1)
        )
        self.conv3 = nn.Conv2d(dw_channel // 2, c, 1)

        ffn_channel = c * ffn_expand
        self.conv4 = nn.Conv2d(c, ffn_channel, 1)
        self.sg2 = SimpleGate()
        self.conv5 = nn.Conv2d(ffn_channel // 2, c, 1)

        self.norm1 = LayerNorm2d(c)
        self.norm2 = LayerNorm2d(c)
        self.beta = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)
        self.gamma = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)

    def forward(self, x):
        res = x
        x = self.norm1(x)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.sg1(x)
        x = x * self.sca(x)
        x = self.conv3(x)
        y = res + x * self.beta

        res = y
        y = self.norm2(y)
        y = self.conv4(y)
        y = self.sg2(y)
        y = self.conv5(y)
        return res + y * self.gamma

class NAFNetSR(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, width=32, enc_blocks=[2, 2], middle_blocks=4, dec_blocks=[2, 2], upscale=2):
        super().__init__()
        self.upscale = upscale
        self.intro = nn.Conv2d(in_channels, width, 3, padding=1)

        self.encoders = nn.ModuleList()
        self.downs = nn.ModuleList()
        curr_width = width
        for n in enc_blocks:
            self.encoders.append(nn.Sequential(*[NAFBlock(curr_width) for _ in range(n)]))
            self.downs.append(nn.Conv2d(curr_width, curr_width * 2, 2, stride=2))
            curr_width *= 2

        self.middle = nn.Sequential(*[NAFBlock(curr_width) for _ in range(middle_blocks)])

        self.decoders = nn.ModuleList()
        self.ups = nn.ModuleList()
        for n in dec_blocks:
            self.ups.append(nn.Sequential(
                nn.Conv2d(curr_width, curr_width * 2, 1),
                nn.PixelShuffle(2)
            ))
            curr_width = curr_width // 2
            self.decoders.append(nn.Sequential(*[NAFBlock(curr_width) for _ in range(n)]))

        self.sr_upsample = nn.Sequential(
            nn.Conv2d(curr_width, curr_width * (upscale ** 2), 3, padding=1),
            nn.PixelShuffle(upscale),
            NAFBlock(curr_width)
        )
        self.ending = nn.Conv2d(curr_width, out_channels, 3, padding=1)

    def forward(self, x):
        feats = self.intro(x)
        skips = []
        for encoder, down in zip(self.encoders, self.downs):
            feats = encoder(feats)
            skips.append(feats)
            feats = down(feats)

        feats = self.middle(feats)

        for up, decoder in zip(self.ups, self.decoders):
            feats = up(feats)
            feats = feats + skips.pop()
            feats = decoder(feats)

        feats = self.sr_upsample(feats)
        return self.ending(feats)

model = NAFNetSR(in_channels=1, out_channels=1, width=32).to(device)
print(f"[INFO] NAFNet Initialized. Total Trainable Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Loss Function & Evaluation Metrics

In [ ]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target)**2 + self.eps**2))

def calculate_psnr(pred: torch.Tensor, target: torch.Tensor, max_val: float = 1.0) -> float:
    pred = torch.clamp(pred, 0.0, max_val)
    target = torch.clamp(target, 0.0, max_val)
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return float('inf')
    return 20.0 * math.log10(max_val) - 10.0 * math.log10(mse)

criterion = CharbonnierLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

## 6. Training & Validation Loop

In [ ]:
train_losses, val_losses, val_psnrs = [], [], []
best_psnr = 0.0
os.makedirs("checkpoints", exist_ok=True)

print("[INFO] Starting Training Loop...")
for epoch in range(1, EPOCHS + 1):
    # Train Epoch
    model.train()
    running_loss = 0.0
    for lr_imgs, gt_imgs in train_loader:
        lr_imgs, gt_imgs = lr_imgs.to(device), gt_imgs.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            pred_imgs = model(lr_imgs)
            loss = criterion(pred_imgs, gt_imgs)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * lr_imgs.size(0)
        
    scheduler.step()
    train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(train_loss)
    
    # Validation Epoch
    model.eval()
    val_loss, val_psnr = 0.0, 0.0
    with torch.no_grad():
        for lr_imgs, gt_imgs in val_loader:
            lr_imgs, gt_imgs = lr_imgs.to(device), gt_imgs.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                pred_imgs = model(lr_imgs)
                loss = criterion(pred_imgs, gt_imgs)
            val_loss += loss.item() * lr_imgs.size(0)
            for p, g in zip(pred_imgs, gt_imgs):
                val_psnr += calculate_psnr(p, g)
                
    val_loss /= len(val_loader.dataset)
    val_psnr /= len(val_loader.dataset)
    val_losses.append(val_loss)
    val_psnrs.append(val_psnr)
    
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save(model.state_dict(), "checkpoints/nafnet_best.pth")
        
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Val PSNR: {val_psnr:.2f} dB (Best: {best_psnr:.2f} dB)")

## 7. Visualization & Performance Curves

In [ ]:
# Plot Loss & PSNR Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, EPOCHS + 1), train_losses, label="Train Loss", color='blue')
axes[0].plot(range(1, EPOCHS + 1), val_losses, label="Val Loss", color='orange')
axes[0].set_title("Charbonnier Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(range(1, EPOCHS + 1), val_psnrs, label="Val PSNR", color='green')
axes[1].set_title("Validation PSNR (dB)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].grid(True)
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Qualitative Results Comparison
Side-by-side comparison of **Input (NoisyLR 128x128)**, **Restored Output (NAFNet 256x256)**, and **Ground Truth (GT 256x256)**.

In [ ]:
model.eval()
with torch.no_grad():
    lr_batch, gt_batch = next(iter(val_loader))
    lr_batch, gt_batch = lr_batch.to(device), gt_batch.to(device)
    pred_batch = model(lr_batch)

num_display = min(3, lr_batch.size(0))
fig, axes = plt.subplots(num_display, 3, figsize=(15, 5 * num_display))

for i in range(num_display):
    lr_img = lr_batch[i, 0].cpu().numpy()
    pred_img = torch.clamp(pred_batch[i, 0], 0.0, 1.0).cpu().numpy()
    gt_img = gt_batch[i, 0].cpu().numpy()
    
    psnr_score = calculate_psnr(pred_batch[i], gt_batch[i])
    
    ax_row = axes[i] if num_display > 1 else axes
    ax_row[0].imshow(lr_img, cmap='gray')
    ax_row[0].set_title(f"Sample {i+1}: Input NoisyLR (128x128)")
    ax_row[0].axis('off')
    
    ax_row[1].imshow(pred_img, cmap='gray')
    ax_row[1].set_title(f"Restored NAFNet (256x256)\nPSNR: {psnr_score:.2f} dB")
    ax_row[1].axis('off')
    
    ax_row[2].imshow(gt_img, cmap='gray')
    ax_row[2].set_title(f"Ground Truth GT (256x256)")
    ax_row[2].axis('off')

plt.tight_layout()
plt.show()